# فصل 10: پردازش زبان طبیعی مکانی

کد این نوت‌بوک عیناً از فصل 10 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۱۰-۴) ژئوکدینگ: از نام مکان تا مختصات


In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="my_geoai_app")

# respect the 1 request/second limit of the public service
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

place = "Sarpol-e Zahab, Iran"      # always append the country
loc = geocode(place)
if loc:
    print(loc.latitude, loc.longitude)

## ۱۰-۶) کارگاه ۱: نقشه‌ی بحران از پیام‌های انگلیسی


In [ ]:
!pip install spacy geopy folium
!python -m spacy download en_core_web_sm

import spacy, folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

nlp = spacy.load("en_core_web_sm")
geocode = RateLimiter(Nominatim(user_agent="crisis").geocode,
                      min_delay_seconds=1.5)

messages = [
    "Massive damage in Sarpol-e Zahab. We need water urgently.",
    "Roads to Qasr-e Shirin are blocked. Avoid this route.",
    "Hospital in Eslamabad-e Gharb is overflowing. Help needed.",
]

m = folium.Map(location=[34.31, 47.07], zoom_start=8)
for msg in messages:
    doc = nlp(msg)                                 # step 1: parse the text
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):           # step 2: NER
            loc = geocode(ent.text + ", Iran")     # step 3: geocoding
            if loc:                            # step 4: keep the hits
                folium.Marker([loc.latitude, loc.longitude],
                              popup=ent.text).add_to(m)
# step 5: an interactive GIS output
m.save("crisis_map.html")

## ۱۰-۷) کارگاه ۲: ژئوکدینگ خبرهای فارسی


In [ ]:
!pip install hazm transformers torch

from hazm import Normalizer
from transformers import pipeline

normalizer = Normalizer()
# step 1: normalise Arabic letters, half-spaces, digits  (CRUCIAL)
text = normalizer.normalize(raw_persian_text)

# step 2: Persian NER with a pre-trained Transformer
ner = pipeline("ner",
               model="HooshvareLab/bert-fa-base-uncased-ner-peyma",
               aggregation_strategy="simple")
entities = ner(text)

places = [e["word"] for e in entities if e["entity_group"] == "LOC"]
print(places)

## ۱۰-۸) انقلاب مدل‌های زبانی بزرگ در Geo-NLP


In [ ]:
prompt = """Read the text and return ONLY a JSON list of place names.
No explanation.
Text: "بارش شدید در سرپل ذهاب و قصرشیرین رخ داد."
Output:"""

# call any LLM API with temperature=0 for a repeatable answer
# likely output: ["سرپل ذهاب", "قصرشیرین"]

## تکمیل پل: از خروجی پایتون به ArcGIS


In [ ]:
import geopandas as gpd
from shapely.geometry import Point
geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf.to_file("crisis_points.geojson", driver="GeoJSON")